In [ ]:
# TGraph records carry attributes; convert to geometry only when needed.
def _as_topology(value):
    from topologicpy.Topology import Topology
    from topologicpy.Vertex import Vertex
    if isinstance(value, (list, tuple)):
        return [_as_topology(item) for item in value]
    if not isinstance(value, dict) or "index" not in value or "dictionary" not in value:
        return value
    representation = value.get("representation")
    attributes = value["dictionary"]
    if representation is None and "src" not in value:
        representation = Vertex.ByCoordinates(
            attributes.get("x", 0), attributes.get("y", 0), attributes.get("z", 0))
    if representation is None:
        raise ValueError("This graph record has no geometry representation")
    return Topology.SetDictionary(representation, attributes)


def _record_dictionary(value):
    from topologicpy.Topology import Topology
    if isinstance(value, dict) and "dictionary" in value:
        return value["dictionary"]
    return Topology.Dictionary(value)


## How to Connect Remote Objects Using a Graph
Typically, only adjacent topologies are connected by an edge in topologicpy. However, there is a method by wich un-connected/remote objects can be related to each other with an edge (i.e. relationship). This method is called "outposts". Here is how it works:

1. First, you must assign a unique ID to each topology. You can use the Dictionary class to achieve this.
2. Then, you must embed in that dictionary a list of unique IDs of all the other topologies (i.e. outposts) to which this topology should be connected. Save that list under a dictionary key (e.g. "outposts")
3. Create a Cluster of these topologies (or aother types of topology as appropriate)
4. Create a graph of the resulting topology and specify toOutposts = True and also specify the outpusts key that you used (usually "outposts") and the ID key that the ID under which the id of the object is stored (usually "id"). 

In [ ]:
#The following two lines are not needed on your computer
import sys
sys.path.append(r"C:\Users\sarwj\OneDrive - Cardiff University\Documents\GitHub\topologicpy\src")

### 1. Import the needed libraries

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.TGraph import TGraph
from topologicpy.Plotly import Plotly
import random

### 2. Create a 3x3x3 CellComplex to use as an example

In [ ]:
cc = CellComplex.Prism(uSides=3, vSides=3, wSides=3)

### 3. Get the cells of the CellComplex and assign each a unique ID

In [ ]:
cells = Topology.Cells(cc)
for i, cell in enumerate(cells):
    d = Dictionary.ByKeyValue("id", i+1)
    cell = Topology.SetDictionary(cell, d)

### 4. Store the IDs of each cell's adjacent cells in that cell's dictionary

In [ ]:
# Get the cells one more time
cells = Topology.Cells(cc)
# Loop through each cell in the CellComplex
for cell in cells:
    # Get the adjacent cells
    adj_cells = Topology.AdjacentTopologies(cell, cc, topologyType="cell")
    # Initialize an empty list to store the adjacent cells ids
    adj_ids = []
    # Loop through the adjacent cells
    for adj_cell in adj_cells:
        # Get the ID of the adjacent cell
        adj_id = Dictionary.ValueAtKey(_record_dictionary(adj_cell), 'id')
        # Add that ID to the list of adjacent IDs
        adj_ids.append(adj_id)
    # Get the existing dictionary of the cell
    d = _record_dictionary(cell)
    # Add the new key/value pair to that dictionary
    d = Dictionary.SetValueAtKey(d, "outposts", adj_ids)
    # Re-assign this new expanded dictionary to the cell
    cell = Topology.SetDictionary(cell, d)
    # Print the id and the list of outposts for verification
    print(Dictionary.ValueAtKey(d, "id"), ":", Dictionary.ValueAtKey(d, "outposts"))

### 5. Explode the CellComplex and ensure that dictionaries are transferred

In [ ]:
exploded = Topology.Explode(cc, scale=1.5, transferDictionaries=True, typeFilter="cell")
Topology.Show(exploded)

### 6. Move the exploded cells to random positions

In [ ]:
# Get the cells of the exploded cluster
cells = Topology.Cells(exploded)
# Initialize a new empty list of cells
new_cells = []
# Loop through each cell
for cell in cells:
    # Create a random displacement
    dx = random.uniform(-2,2)
    dy = random.uniform(-2,2)
    dz = random.uniform(-2,2)
    # Move (Translate) the cell by a random displacement
    cell = Topology.Translate(_as_topology(cell), _as_topology(dx), _as_topology(dy), _as_topology(dz))
    # Add this newly moved cell to the list of new cells
    new_cells.append(cell)

### 7. Create a new cluster from these new cells

In [ ]:
exploded = Cluster.ByTopologies(_as_topology(new_cells))
Topology.Show(exploded)

### 8. Create a Graph from this new cluster with the outposts method set to True

In [ ]:
graph = TGraph.ByTopology(exploded, toOutposts=True, idKey="id")

### 9. Draw the final result of the exploded cluster and the resulting graph

In [ ]:
data01 = Plotly.DataByGraph(graph, sagitta=0.1, absolute=False, sides=24, edgeColor="red", vertexColor="blue")
data02 = Plotly.DataByTopology(exploded)
figure = Plotly.FigureByData(data01+data02, width=1024, height=800)
Plotly.Show(figure)